# HiveMind Benchmark Autopilot (3 Suites, Foreground + Live Google Drive Sync)

Dieses Notebook führt **alle 3 HiveMind Benchmarks** nacheinander im **Vordergrund** aus, sodass Google Colab die laufende Zelle sieht und **nicht durch `tmux`-Leerlauf in den Idle-Disconnect geht**.

### Was dieses Notebook automatisch erledigt:
1. **Vorher-Sync mit Google Drive:** Mountet Drive gleich am Anfang und stellt vorhandene Checkpoints (`MyDrive/hivemind_bench/results/*.json`) wieder her.
2. **Live-Zwischenstand-Push nach jeder einzelnen Aufgabe:** Nach jeder Task × Harness-Zelle schreibt `runner.py` atomar lokal **und** spiegelt den aktuellen JSON-Stand sofort nach Google Drive (`--sync-dir`).
3. **Nahtloses Auto-Resume (`--resume`):** Falls Colab nach vielen Stunden trennt oder neu startet, einfach Zelle 1 → 2 → 3 erneut ausführen: bereits erledigte `(run, task, condition)`-Zellen werden aus Drive geladen, automatisch übersprungen, und der Lauf macht exakt dort weiter, wo er aufgehört hat.
4. **Identische Fairness-Regeln (`2722e92`):** Gleiches Modell, gleicher `llama-server`, **identisches Timeout** über alle Harnesses (`300s` Core/Polyglot, `1800s` DeepSWE), inkl. echtem Token- & Cache-Tracking für HiveMind, Aider, Hermes (`state.db`) und Pi (`--mode json`).

### Die 3 Benchmark-Phasen im Autopilot:
1. **Core Suite (`23 Tasks`)** — Standardmäßig 3 Runs (69 Zellen/Harness) für statistische Varianz & Wilson 95% CI.
2. **Polyglot Python (`34 Tasks`)** — Offizieller Aider-AI-Corpus.
3. **DeepSWE Real-World (`3 Repo-Tasks @ 128k`)** — `tomlkit`, `httpx`, `cattrs` (`1800s` Timeout).

In [ ]:
# 1) Google Drive mounten & vorhandene Checkpoints prüfen
import json
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_LOGS = DRIVE_ROOT / "logs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive verbunden: {DRIVE_RESULTS}")
except ImportError:
    DRIVE_RESULTS = Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Hinweis: google.colab nicht aktiv, lokaler Backup-Ordner: {DRIVE_RESULTS}")

existing = sorted(DRIVE_RESULTS.glob("*.json"))
if existing:
    print(f"\nGefundene Checkpoints in Drive ({len(existing)}):")
    for p in existing:
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
            print(f"  - {p.name:30s} | model={d.get('model')} ctx={d.get('num_ctx', '?')} rows={len(d.get('results', []))}")
        except Exception as exc:
            print(f"  - {p.name:30s} | (nicht lesbar: {exc})")
else:
    print("\nNoch keine Checkpoints in Drive — frischer Start.")

In [ ]:
# 2) Repo aktualisieren, Drive -> Lokal vorab syncen & Basis-Setup (inkl. Polyglot & DeepSWE)
import shutil, subprocess
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = DRIVE_ROOT / "results" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = DRIVE_ROOT / "logs" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

![ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='[bench] commit: %h %s'

# Vorher-Sync: Alle Drive-Checkpoints in das lokale results/-Verzeichnis ziehen
local_res = Path("/root/HiveMind_llama/bench/results")
local_res.mkdir(parents=True, exist_ok=True)
for src in sorted(DRIVE_RESULTS.glob("*.json")):
    dst = local_res / src.name
    if (not dst.exists()) or src.stat().st_size >= dst.stat().st_size:
        shutil.copy2(src, dst)
        print(f"[pre-sync] {src.name} -> {dst}")

# Basis-Setup (Python 3.14, llama.cpp, Aider, Hermes, Pi + Node 22)
!HM_MODEL="skip" HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup

# Alle 3 Task-Corpora bereithalten (Core ist im Repo, Polyglot & DeepSWE werden importiert)
if not Path("/root/HiveMind_llama/bench/tasks_polyglot/SOURCE.json").is_file():
    subprocess.run(["bash", "-lc", "HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot"], check=True)
else:
    print("[setup] Polyglot Tasks (34) bereits vorhanden.")

subprocess.run(["pip", "install", "-q", "hypothesis", "trio", "attrs", "pytest", "pytest-asyncio", "anyio"], check=False)
if not Path("/root/HiveMind_llama/bench/tasks_deepswe/SOURCE.json").is_file():
    subprocess.run(["bash", "/root/rent_bootstrap.sh", "deepswe"], check=True)
else:
    print("[setup] DeepSWE Tasks (3) bereits vorhanden.")

In [ ]:
# 3) AUTOPILOT STARTEN — läuft im Vordergrund + pusht jeden Task direkt nach Drive
import os, subprocess, sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals() or "DRIVE_LOGS" not in globals():
    _dr = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = (_dr / "results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = (_dr / "logs") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

# ==================== KONFIGURATION ====================
# "auto" = T4 (<20GB VRAM) -> 4B MTP @ 128k | L4/A100 (>=20GB) -> 35B MoE
# Oder explizit "4b" bzw. "35b" setzen:
MODEL_CHOICE = "auto"

# Harnesses:
#   - Standard 4-Harness Arena: "hivemind_duo,aider,hermes,pi"
#   - Inkl. Full Toolset:       "hivemind_duo,hivemind_duo_full,aider,hermes,pi"
CONDITIONS = "hivemind_duo,aider,hermes,pi"

# Welche der 3 Benchmarks sollen laufen (und wie viele Runs jeweils)?
RUN_CORE     = True;  CORE_RUNS     = 3   # Benchmark 1: Core Suite (23 Tasks)
RUN_POLYGLOT = True;  POLYGLOT_RUNS = 1   # Benchmark 2: Polyglot Python (34 Tasks)
RUN_DEEPSWE  = True;  DEEPSWE_RUNS  = 1   # Benchmark 3: DeepSWE (3 Repo-Tasks @ 128k)

# Kontext für Core/Polyglot beim 35B-Modell (32768 oder 65536); 4B nutzt immer 131072 (128k):
CTX_35B_SMALL_SUITES = 32768
# =======================================================

def _detect_vram_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        return float(out) / 1024.0
    except Exception:
        return 15.0

vram_gb = _detect_vram_gb()
use_model = MODEL_CHOICE.strip().lower()
if use_model == "auto":
    use_model = "4b" if vram_gb < 20.0 else "35b"

if use_model == "4b":
    model_id = "qwen3.5:4b-mtp"
    tag = "4b"
    ctx_small = 131072
    ctx_deepswe = 131072
else:
    model_id = "qwen3.6:35b-a3b-ud"
    tag = "35b"
    ctx_small = CTX_35B_SMALL_SUITES
    ctx_deepswe = 131072

stages = []
if RUN_CORE:
    stages.append({
        "name": f"core_{tag}_{ctx_small // 1024}k",
        "tasks": "bench/tasks",
        "model": model_id,
        "ctx": ctx_small,
        "runs": CORE_RUNS,
        "cell_timeout": 900,
    })
if RUN_POLYGLOT:
    stages.append({
        "name": f"polyglot_{tag}_{ctx_small // 1024}k",
        "tasks": "bench/tasks_polyglot",
        "model": model_id,
        "ctx": ctx_small,
        "runs": POLYGLOT_RUNS,
        "cell_timeout": 900,
    })
if RUN_DEEPSWE:
    stages.append({
        "name": f"deepswe_{tag}_{ctx_deepswe // 1024}k",
        "tasks": "bench/tasks_deepswe",
        "model": model_id,
        "ctx": ctx_deepswe,
        "runs": DEEPSWE_RUNS,
        "cell_timeout": 1800,
    })

print(f"=== AUTOPILOT GESTARTET (GPU VRAM: {vram_gb:.1f} GB -> Modus: {use_model.upper()} / {model_id}) ===")
print(f"Conditions: {CONDITIONS}")
print(f"Drive Sync: {DRIVE_RESULTS}")
print(f"Geplante Stages ({len(stages)}): {[s['name'] for s in stages]}\n")

for idx, st in enumerate(stages, 1):
    out_file = Path(f"/root/HiveMind_llama/bench/results/{st['name']}.json")
    log_file = DRIVE_LOGS / f"{st['name']}.log"
    print(f"\n=================================================================")
    print(f"[STAGE {idx}/{len(stages)}] {st['name']} | tasks={st['tasks']} | runs={st['runs']} | ctx={st['ctx']}")
    print(f"=================================================================", flush=True)

    env = os.environ.copy()
    env.update({
        "HM_MODEL": st["model"],
        "HM_CTX": str(st["ctx"]),
        "HM_TASKS": st["tasks"],
        "HM_RUNS": str(st["runs"]),
        "HM_RUN_OFFSET": "0",
        "HM_CONDITIONS": CONDITIONS,
        "HM_CELL_TIMEOUT": str(st["cell_timeout"]),
        "HM_OUT": str(out_file),
        "HM_SYNC_DIR": str(DRIVE_RESULTS),
        "HM_RESUME": "1",
    })

    with open(log_file, "a", encoding="utf-8") as lf:
        lf.write(f"\n=== STAGE {st['name']} START ===\n")
        proc = subprocess.Popen(
            ["bash", "/root/rent_bootstrap.sh", "fg"],
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        try:
            assert proc.stdout is not None
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            rc = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait(timeout=10)
            raise
        if rc != 0:
            raise RuntimeError(f"Stage {st['name']} abgebrochen (exit={rc}). Bisherige Ergebnisse sind in {DRIVE_RESULTS} gespeichert!")

print(f"\nALLE {len(stages)} BENCHMARK-STAGES ABGESCHLOSSEN! Ergebnisse liegen unter: {DRIVE_RESULTS}")

In [ ]:
# 4) Leaderboard über alle in Google Drive gespeicherten Benchmarks
import glob, subprocess
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_RESULTS = Path("/content/drive/MyDrive/hivemind_bench/results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results")

files = sorted(glob.glob(str(DRIVE_RESULTS / "*.json")))
if not files:
    print("Noch keine JSON-Ergebnisse gefunden.")
else:
    for f in files:
        print(f"\n==================== {Path(f).name} ====================")
        subprocess.run(
            ["/root/HiveMind_llama/.venv/bin/python", "/root/HiveMind_llama/bench/analyze.py", "--results", f, "--by-model"],
            check=False,
        )

In [ ]:
# 5) Server & VRAM sauber freigeben
!bash /root/rent_bootstrap.sh stop